# Stage 3 — concat576 + LoRA (official)

Simple visual prefix (no QGCRF, no slot compression):

```
[576 Projection-B globals] + [K × 16 Projection-A regions] + format / Question / Answer
```

| Mode | Regions |
|------|---------|
| **global576** | omitted (`include_regions=False`) |
| **concat576** | up to K boxes × 16 tokens |

**Trainable:** LoRA only. ViT, Projection-A, Projection-B frozen.

Implementation: `stage3_train.py`, `training.py`, `inference.py`, `config.Stage3Config`.


In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


In [ ]:
import os
from pathlib import Path

hf_cache_root = Path(os.environ['REVA_HF_CACHE_ROOT']).expanduser()
checkpoint_root = Path(os.environ['REVA_CHECKPOINT_ROOT']).expanduser()
hf_cache_root.mkdir(parents=True, exist_ok=True)
checkpoint_root.mkdir(parents=True, exist_ok=True)

# Configuring environment variables
os.environ['HF_HOME'] = str(hf_cache_root)
os.environ['HF_DATASETS_CACHE'] = str(hf_cache_root / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(hf_cache_root / 'hub')
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_TORCH", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print('Free in /tmp :', end=' ')
!df -h /tmp | awk 'NR==2{print $4}'

## 0. Dependencies


In [ ]:
import importlib
import subprocess
import sys


def _ok(mod, attr=None):
    try:
        m = importlib.import_module(mod)
        return True if attr is None else hasattr(m, attr)
    except Exception:
        return False


if not _ok("torch", "Tensor"):
    raise RuntimeError(
        "Broken PyTorch. Restore with conda (do not pip install torch from this notebook)."
    )

to_install = []
if not _ok("peft"):
    to_install.append("peft>=0.14.0")
if not _ok("transformers"):
    to_install.append("transformers>=4.48.0")
if not _ok("accelerate"):
    to_install.append("accelerate>=0.34.0")
import numpy as np
if int(np.__version__.split(".")[0]) >= 2:
    to_install.append("numpy<2.0")

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install, "-q"])
    print("Installed:", ", ".join(to_install))
    print("Restart kernel, then continue from section 1.")
else:
    import torch, transformers, peft
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    print("transformers", transformers.__version__, "peft", peft.__version__)

## 1. Config


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path(".").resolve()
CLEAN_PKL = Path(os.environ.get("REVA_STAGE3_CLEAN_PKL", Path(os.environ["REVA_DECONTAMINATION_ROOT"]) / "stage3_eval_clean.pkl"))
PROJECTION_B_WEIGHTS = PROJECT_DIR / "projection_b_best_weights.pt"
PROJECTION_A_WEIGHTS = PROJECT_DIR / "projection_a_curriculum_best_weights.pt"

# Optional: resume / warm-start LoRA (e.g. sandbox best_lora)
LORA_RESUME_DIR = None  # Path(os.environ["REVA_CHECKPOINT_ROOT"]) / "concat576_sandbox_.../best_lora"

RUN_TRAINING = True
RUN_EVAL = True
MAX_TRAIN_SAMPLES = None          # e.g. 2000 for smoke
MAX_OPTIMIZER_STEPS = 10000
MAX_BOXES = 20                    # K → 576 + 20*16 = 896 visual tokens
EVAL_N = 2000
EVAL_SEED = 42

GQA_ROOT = Path(os.environ["REVA_GQA_ROOT"])
VQAV2_ROOT = Path(os.environ["REVA_VQAV2_ROOT"])
COCO_ROOT = Path(os.environ.get("REVA_COCO_ROOT", os.environ.get("REVA_COCO_DIR", Path(os.environ["REVA_REGION_DATA_ROOT"]) / "coco")))
VISUAL7W_ROOT = Path(os.environ["REVA_VISUAL7W_ROOT"])

TAG = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(os.environ["REVA_CHECKPOINT_ROOT"]) / f"concat576_stage3_{TAG}"

assert CLEAN_PKL.exists(), CLEAN_PKL
assert PROJECTION_B_WEIGHTS.exists(), PROJECTION_B_WEIGHTS
assert PROJECTION_A_WEIGHTS.exists(), PROJECTION_A_WEIGHTS
print("Output:", OUTPUT_DIR)
print(f"Visual prefix ≤ {576 + MAX_BOXES * 16} tokens")

## 2. Load stack + LoRA


In [ ]:
from reva.config import Stage3Config
from reva.models import (
    build_region_feature_extractor,
    load_frozen_vit,
    load_projection_b_weights,
    load_qwen_with_lora,
    load_region_feature_extractor,
)
from pathlib import Path

config = Stage3Config(
    clean_pkl_path=CLEAN_PKL,
    projection_b_path=str(PROJECTION_B_WEIGHTS),
    projection_a_weights_path=str(PROJECTION_A_WEIGHTS.name),
    checkpoint_dir=PROJECTION_A_WEIGHTS.parent,
    lora_output_dir=OUTPUT_DIR,
    training_log_path=OUTPUT_DIR / "training_log.json",
    max_boxes_per_image=MAX_BOXES,
    max_optimizer_steps=MAX_OPTIMIZER_STEPS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    dataloader_num_workers=0,
)

frozen_vit, clip_image_processor = load_frozen_vit(config)
projection_head_b = load_projection_b_weights(str(PROJECTION_B_WEIGHTS), config)
region_extractor = build_region_feature_extractor(config)
load_region_feature_extractor(region_extractor, str(PROJECTION_A_WEIGHTS), config)

qwen, qwen_tokenizer = load_qwen_with_lora(config)

if LORA_RESUME_DIR is not None and Path(LORA_RESUME_DIR).exists():
    from peft import PeftModel
    base = qwen.get_base_model() if hasattr(qwen, "get_base_model") else qwen
    qwen = PeftModel.from_pretrained(base, str(LORA_RESUME_DIR), is_trainable=True)
    qwen.config.use_cache = False
    qwen.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    qwen.enable_input_require_grads()
    print("Resumed LoRA from", LORA_RESUME_DIR)

qwen.config.use_cache = False
qwen.print_trainable_parameters()
print("device:", config.device, "| dtype:", config.compute_dtype)

## 3. Train LoRA on concat576


In [ ]:
from reva.stage3_train import load_stage3_clean_pool
from reva.training import train_stage3_lora

train_samples = load_stage3_clean_pool(CLEAN_PKL, max_samples=MAX_TRAIN_SAMPLES)

if RUN_TRAINING:
    qwen, training_log = train_stage3_lora(
        qwen,
        qwen_tokenizer,
        frozen_vit,
        projection_head_b,
        region_extractor,
        train_samples,
        clip_image_processor,
        config,
    )
    print("Best LoRA:", config.lora_output_dir / config.lora_best_subdir)
else:
    print("Skipping training")

## 4. Load best LoRA for eval


In [ ]:
from peft import PeftModel

best_lora = config.lora_output_dir / config.lora_best_subdir
assert best_lora.exists(), f"Missing {best_lora}"

base_qwen = qwen.get_base_model() if hasattr(qwen, "get_base_model") else qwen
qwen_eval = PeftModel.from_pretrained(base_qwen, str(best_lora), is_trainable=False)
qwen_eval.eval()
frozen_vit.eval()
projection_head_b.eval()
region_extractor.eval()
print("Loaded", best_lora)

## 5. Eval — global576 vs concat576 (n=EVAL_N)


In [ ]:
import json
import random
from pathlib import Path

from tqdm.auto import tqdm
from reva.evaluation import (
    download_coco_val2014_instances,
    download_gqa_val,
    download_vqav2_val,
    _vqav2_image_path,
)
from reva.inference import normalise_answer, run_concat576_inference, vqa_soft_score
from reva.stage3_dataset import (
    DEFAULT_FORMAT_PROMPT,
    _cap_boxes,
    _iter_visual7w_qa_pairs,
    _resolve_gqa_scene_dir,
    _resolve_visual7w_dataset_json,
    _visual7w_box_to_xyxy,
    _visual7w_boxes_by_id,
    load_gqa_scene_graphs,
)


def clean_pred(t):
    return t.strip().split("\n")[0].strip().rstrip(".")


def score(pred, sample):
    if "answers" in sample:
        return vqa_soft_score(pred, sample["answers"])
    return float(normalise_answer(pred) == normalise_answer(sample["answer"]))


def eval_bench(samples, boxes_fn, n, desc):
    rng = random.Random(EVAL_SEED)
    pool = samples[:]
    rng.shuffle(pool)
    pool = pool[: min(n, len(pool))]
    g = c = 0.0
    for s in tqdm(pool, desc=desc):
        fmt = s.get("format_prompt", DEFAULT_FORMAT_PROMPT)
        boxes = boxes_fn(s)
        pg, _ = run_concat576_inference(
            s["image_path"], s["question"],
            boxes_px=[], include_regions=False,
            frozen_vit=frozen_vit, projection_head_b=projection_head_b,
            region_extractor=region_extractor, qwen=qwen_eval,
            qwen_tokenizer=qwen_tokenizer, clip_image_processor=clip_image_processor,
            config=config, format_prompt=fmt,
        )
        pc, _ = run_concat576_inference(
            s["image_path"], s["question"],
            boxes_px=boxes, include_regions=True,
            frozen_vit=frozen_vit, projection_head_b=projection_head_b,
            region_extractor=region_extractor, qwen=qwen_eval,
            qwen_tokenizer=qwen_tokenizer, clip_image_processor=clip_image_processor,
            config=config, format_prompt=fmt,
        )
        g += score(clean_pred(pg), s)
        c += score(clean_pred(pc), s)
    n_eff = len(pool)
    return {
        "global576_pct": 100 * g / n_eff,
        "concat576_pct": 100 * c / n_eff,
        "delta_pp": 100 * (c - g) / n_eff,
        "n": n_eff,
    }


def build_v7w(v7w_dir, split="val"):
    v7w_dir = Path(v7w_dir)
    dj = _resolve_visual7w_dataset_json(v7w_dir)
    if dj is None:
        return []
    with open(dj) as f:
        data = json.load(f)
    boxes_by_id = _visual7w_boxes_by_id(data)
    out = []
    for qa, img_meta in _iter_visual7w_qa_pairs(data, split=split):
        qa_type = str(qa.get("type", qa.get("qa_type", ""))).lower()
        if qa_type and qa_type not in ("which", "where"):
            continue
        rel = img_meta.get("filename", img_meta.get("image_path", qa.get("image_path", "")))
        if not rel:
            continue
        img = v7w_dir / rel
        if not img.exists():
            img = dj.parent / rel
        if not img.exists():
            img = v7w_dir / "visual7w_pointing/images" / Path(rel).name
        if not img.exists():
            continue
        boxes, answer = [], str(qa.get("multiple_choice_answer", qa.get("answer_text", qa.get("answer", ""))))
        if qa.get("answer") is not None:
            try:
                ab = boxes_by_id.get(int(qa["answer"]))
            except (TypeError, ValueError):
                ab = None
            if ab is not None:
                boxes = [_visual7w_box_to_xyxy(ab)]
                answer = str(ab.get("name", answer))
        q = qa.get("question", qa.get("qa_text", qa.get("question_text", "")))
        if boxes and answer and q:
            out.append({"image_path": str(img), "question": str(q), "answer": str(answer), "boxes": boxes})
    print(f"Visual7W candidates: {len(out):,}")
    return out


if RUN_EVAL:
    results = {}

    gqa_samples, gqa_img = download_gqa_val(GQA_ROOT)
    scene = load_gqa_scene_graphs(_resolve_gqa_scene_dir(GQA_ROOT) / "val_sceneGraphs.json")

    def gqa_boxes(s):
        sg = scene.get(str(s["image_id"]), {})
        w, h = float(sg.get("width", 0) or 0), float(sg.get("height", 0) or 0)
        boxes = []
        for obj in (sg.get("objects") or {}).values():
            x, y, bw, bh = float(obj["x"]), float(obj["y"]), float(obj["w"]), float(obj["h"])
            if bw > 0 and bh > 0:
                boxes.append(
                    [x, y, x + bw, y + bh]
                    if max(x, y, bw, bh) > 1
                    else [x * w, y * h, (x + bw) * w, (y + bh) * h]
                )
        return _cap_boxes(boxes, MAX_BOXES)

    gqa_eval = [
        {
            "image_path": str(gqa_img / f"{s['image_id']}.jpg"),
            "image_id": s["image_id"],
            "question": s["question"],
            "answer": s["answer"],
        }
        for s in gqa_samples
    ]
    results["gqa"] = eval_bench(gqa_eval, gqa_boxes, EVAL_N, f"GQA n={EVAL_N}")

    vqa_samples, vqa_img = download_vqav2_val(VQAV2_ROOT)
    coco = json.load(open(download_coco_val2014_instances(COCO_ROOT)))
    by_img = {}
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        by_img.setdefault(ann["image_id"], []).append([x, y, x + w, y + h])
    vqa_eval = [
        {
            "image_path": str(_vqav2_image_path("val", vqa_img, s["image_id"])),
            "question": s["question"],
            "answers": s["answers"],
            "image_id": s["image_id"],
        }
        for s in vqa_samples
    ]
    results["vqav2"] = eval_bench(
        vqa_eval,
        lambda s: _cap_boxes(by_img.get(s["image_id"], []), MAX_BOXES),
        EVAL_N,
        f"VQAv2 n={EVAL_N}",
    )

    v7w = build_v7w(VISUAL7W_ROOT)
    results["visual7w"] = eval_bench(
        v7w,
        lambda s: _cap_boxes(s.get("boxes", []), MAX_BOXES),
        min(EVAL_N, len(v7w)),
        "Visual7W",
    )

    print("\n=== Stage 3 concat576 ===")
    for b, r in results.items():
        print(
            f"{b:10s} global576={r['global576_pct']:.2f}%  "
            f"concat576={r['concat576_pct']:.2f}%  "
            f"delta={r['delta_pp']:+.2f}pp  n={r['n']}"
        )
    mean_delta = sum(r["delta_pp"] for r in results.values()) / len(results)
    payload = {
        "experiment": "concat576_stage3",
        "max_boxes": MAX_BOXES,
        "mean_delta_pp": mean_delta,
        "results": results,
        "output_dir": str(config.lora_output_dir),
    }
    path = config.lora_output_dir / f"concat576_eval_n{EVAL_N}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    json.dump(payload, open(path, "w"), indent=2)
    print(f"\nMean delta: {mean_delta:+.2f} pp")
    print("Saved:", path)
else:
    print("Skipping eval")

Download Grounding DINO once if missing:
```bash
pip install groundingdino-py
mkdir -p "${REVA_GROUNDING_DINO_ROOT:-$HOME/reva-data/groundingdino}" && cd "${REVA_GROUNDING_DINO_ROOT:-$HOME/reva-data/groundingdino}"
wget -c https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
wget -c https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py
```
Download RAM++ once if missing (relies on a specific local fork or an unreleased version from GitHub):
```bash
pip install git+https://github.com
pip install recognize-anything
pip install fairscale

Or:

pip install git+https://github.com/xinyu1205/recognize-anything.git
pip install fairscale
```

In [ ]:
from pathlib import Path
from reva.evaluation import load_stage3_eval_stack
from reva.inference import run_combined_inference, parse_manual_boxes
from reva.stage3_dataset import DEFAULT_FORMAT_PROMPT

WEIGHTS_DIR = Path(os.environ.get("REVA_WEIGHTS_DIR", "weights"))
PROJECTION_B = WEIGHTS_DIR / "projection_b_best_weights.pt"
PROJECTION_A = WEIGHTS_DIR / "projection_a_curriculum_best_weights.pt"
LORA = WEIGHTS_DIR / "lora_adapter_weights/stage3_lora"

BOX_SOURCE = "ram"  # hybrid / question / ram
MAX_BOXES = 20
LOAD_RAM = True
LIMIT = None                
RESUME = True # False deletes selected benchmark's old JSONL

stack = load_stage3_eval_stack(
    projection_b_path=PROJECTION_B,
    projection_a_path=PROJECTION_A,
    lora_path=LORA,
    max_boxes=MAX_BOXES,
    box_source=BOX_SOURCE,
    load_ram=True,
)

print("Ready on", stack.config.device)

In [ ]:
#### from pathlib import Path
import importlib
import reva.models as models

IMAGES_ROOT = Path(os.environ.get("REVA_TEST_IMAGES_ROOT", "my_test_images"))
FORMAT_PROMPT = "Answer in a single word or short phrase."

models = importlib.reload(models)
models._apply_transformers_compat_shim()
models._rebind_grounding_dino_bert_methods(stack.grounding_dino.model)

def ask_stage3(
    image_path_or_url: str,
    question: str,
    *,
    manual_boxes=None,
    text_prompt=None,
    show_boxes=True,
    max_new_tokens=32,
):
    result = run_combined_inference(
        image_path_or_url,
        question,
        frozen_vit=stack.frozen_vit,
        projection_head_b=stack.projection_head_b,
        region_extractor=stack.region_extractor,
        frozen_qwen=stack.qwen,
        qwen_tokenizer=stack.tokenizer,
        clip_image_processor=stack.image_processor,
        config=stack.config,
        grounding_dino=stack.grounding_dino,
        ram_proposer=stack.ram_proposer,
        manual_boxes=manual_boxes,
        text_prompt=text_prompt,
        box_source=stack.config.box_source,
        format_prompt=FORMAT_PROMPT,
        show_boxes=show_boxes,
        max_new_tokens=max_new_tokens,
        verbose=True,
        include_regions=True,
    )
    print("\nAnswer:", result["answer"])
    print(
        f"Prefix: {result['global_token_count']} global + "
        f"{result['num_region_tokens']} region "
        f"({result['num_regions']} boxes)"
    )
    if result.get("labels"):
        print("Labels:", result["labels"])
    if result.get("grounding_prompt"):
        print("Grounding:", result["grounding_prompt"])
    return result


while True:
    image_input = input("\nImage path/URL/filename (or quit): ").strip()
    if image_input.lower() in {"quit", "q", "exit"}:
        break

    if image_input.startswith("http") or Path(image_input).is_file():
        image_path = image_input
    else:
        image_path = str(IMAGES_ROOT / image_input)

    box_str = input(
        "Manual boxes x1,y1,x2,y2 (; separated), or Enter for hybrid proposer: "
    ).strip()
    manual_boxes = parse_manual_boxes(box_str) if box_str else None

    text_prompt = None
    if manual_boxes is None:
        custom = input(
            "Optional grounding prompt (Enter = auto from question / RAM++): "
        ).strip()
        text_prompt = custom or None

    question = input("Question (or quit): ").strip()
    if question.lower() in {"quit", "q", "exit"}:
        break
    if not question:
        continue

    ask_stage3(
        image_path,
        question,
        manual_boxes=manual_boxes,
        text_prompt=text_prompt,
        show_boxes=True,
    )